# 13주차 과제 · 서울 공공자전거 대여량 예측

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phyzikyh/computing/blob/main/%EA%B0%95%EC%9D%98%EA%B5%90%EC%9E%AC/week13_seoul_bike_assignment.ipynb)

Colab에서는 한글 글꼴이 기본으로 없으므로 다음 두 줄을 **먼저 실행한 뒤 런타임을 다시 시작**합니다.

```
!apt-get -qq -y install fonts-nanum
!fc-cache -fv
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

fonts = {font.name for font in fm.fontManager.ttflist}
for name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if name in fonts:
        plt.rcParams["font.family"] = name
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 20)

In [ ]:
URL = "https://raw.githubusercontent.com/phyzikyh/computing/main/%EA%B0%95%EC%9D%98%EA%B5%90%EC%9E%AC/data/SeoulBikeData.csv"


def find_file(name):
    starts = [Path.cwd()]
    notebook = globals().get("__vsc_ipynb_file__")
    if notebook:
        starts.insert(0, Path(notebook).parent)
    for start in starts:
        for folder in [start, *start.parents][:4]:
            for sub in ["", "data", "강의교재/data"]:
                path = folder / sub / name
                if path.exists():
                    return path
    return None


csv_path = find_file("SeoulBikeData.csv")
source = csv_path if csv_path else URL
bike = pd.read_csv(source, encoding="latin1")

In [ ]:
bike = bike.rename(columns={
    "Date": "date", "Rented Bike Count": "rented_count", "Hour": "hour",
    "Temperature(°C)": "temperature", "Humidity(%)": "humidity",
    "Wind speed (m/s)": "wind_speed", "Visibility (10m)": "visibility",
    "Dew point temperature(°C)": "dew_point", "Solar Radiation (MJ/m2)": "solar_radiation",
    "Rainfall(mm)": "rainfall", "Snowfall (cm)": "snowfall",
    "Seasons": "season", "Holiday": "holiday", "Functioning Day": "functioning_day"
})
bike["date"] = pd.to_datetime(bike["date"], format="%d/%m/%Y")
operating = bike[bike["functioning_day"] == "Yes"].copy()

# 요일 숫자(월요일 0 ~ 일요일 6)와 비가 왔는지 여부(1 또는 0)를 준비합니다.
operating["weekday_number"] = operating["date"].dt.dayofweek
operating["is_rain"] = (operating["rainfall"] > 0).astype(int)
print("운영한 시간의 기록:", len(operating), "행")
operating.head(3)

## 분석에 사용할 열

한 행은 **서울 전체의 한 날짜·한 시간에 대한 기록**입니다. 개인 한 명이나 대여소 하나의 기록이 아닙니다. 예측하려는 값은 그 한 시간의 대여 건수 `rented_count`입니다.

| 열 | 의미 |
|---|---|
| `date`, `hour` | 날짜와 시각(0~23시) |
| `rented_count` | 그 한 시간의 대여 건수 |
| `temperature` | 기온(℃) |
| `humidity` | 습도(%) |
| `wind_speed` | 풍속(m/s) |
| `solar_radiation` | 일사량(MJ/m²) |
| `rainfall`, `is_rain` | 강수량(mm)과 비가 왔는지 여부(`1` 또는 `0`) |
| `snowfall` | 적설량(cm) |
| `weekday_number` | 요일 숫자(월요일 0 ~ 일요일 6) |

모델에는 숫자 열만 입력으로 넣을 수 있습니다. 자료가 없는 비운영 시간은 제외한 `operating`을 사용합니다.

## 문제 1 · 예측 문제 정의하기

1. 이 과제에서 **목표 변수**는 무엇이고, 한 행은 무엇을 뜻하는지 쓰세요.
2. `operating`의 행 수와 `rented_count`의 평균, 최솟값, 최댓값을 구하세요.
3. 입력 후보 `temperature`, `humidity`, `wind_speed`, `solar_radiation`, `rainfall`, `snowfall`, `hour` 중 대여량과 관계가 깊을 것 같은 세 가지를 **결과를 보기 전에** 고르고 이유를 쓰세요.

힌트: `len()`, `describe()`.

In [ ]:
# 문제 1 · 코드 작성 1

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 2 · 학습용과 평가용으로 나누고 기준 만들기

모델이 쓸모 있는지 판단하려면 비교할 기준이 필요합니다. 입력을 쓰지 않고 학습용의 평균 하나로 모든 시간을 예측하는 가장 단순한 방법을 **기준 예측**으로 삼습니다. 모델의 평균 오차가 이 기준 예측의 오차보다 작아야 쓸모가 있습니다.

1. 입력 변수 `X`를 `temperature`, `hour`, `humidity`, `is_rain` 네 열(이 순서)로, 목표 변수 `y`를 `rented_count`로 만드세요. 열 이름 목록은 `features`에 저장합니다.
2. `train_test_split`으로 `test_size=0.2`, `random_state=0`으로 나누어 `X_train`, `X_test`, `y_train`, `y_test`를 만들고 각 개수를 출력하세요.
3. 입력을 쓰지 않고 **학습용 평균 대여량**으로만 평가용 모든 시간을 예측했을 때의 평균 절대 오차를 구해 `baseline_error`에 저장하세요.
4. 이 값은 앞으로 만들 모델의 어떤 기준이 되는지 한 문장으로 쓰세요.

힌트: `train_test_split(X, y, test_size=0.2, random_state=0)`, `mean_absolute_error(실제값, 예측값)`, 평균을 평가용 개수만큼 반복한 목록 `[평균] * len(y_test)`.

**확인값** 학습용 6,772개, 평가용 1,693개, 학습용 평균 734.6건, `baseline_error` 513.4건

In [ ]:
# 문제 2 · 코드 작성 1

In [ ]:
# 문제 2 · 코드 작성 2

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 3 · 기온으로 대여량 예측하기

1. `LinearRegression`을 학습용의 `temperature` 열 하나로 학습시키고, 기울기와 절편을 출력하세요.
2. 기온이 20℃일 때와 30℃일 때의 예측 대여량을 구하세요.
3. 학습용 자료의 산점도 위에 모델이 찾은 직선을 그리세요. 제목과 축 이름을 붙이세요.
4. 평가용 자료의 평균 오차를 구해 `baseline_error`와 비교하세요. 기울기를 "기온이 1℃ 오르면 ~"로 시작하는 문장으로 해석하세요.

힌트: `X_train[["temperature"]]`(이중 대괄호), `fit()`, `coef_`, `intercept_`, `predict(pd.DataFrame({"temperature": [20, 30]}))`.
직선은 기온의 최솟값과 최댓값 두 점의 예측값을 이어서 그립니다.

```
line_x = pd.DataFrame({"temperature": [X_train["temperature"].min(), X_train["temperature"].max()]})
plt.scatter(X_train["temperature"], y_train, s=8, alpha=0.3)
plt.plot(line_x["temperature"], model.predict(line_x), color="red")
```

**확인값** 기울기 30.12, 절편 348.9, 평균 오차 393.6건

In [ ]:
# 문제 3 · 코드 작성 1

In [ ]:
# 문제 3 · 코드 작성 2

In [ ]:
# 문제 3 · 코드 작성 3

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 4 · 입력 변수를 늘려 보기

1. 네 입력(`X_train`의 모든 열)으로 선형 회귀를 학습시키고, 열 이름과 계수를 짝지어 출력하세요.
2. 평가용 평균 오차를 구해 문제 3의 값과 비교하세요.
3. `is_rain`의 계수를 "다른 입력이 같을 때 비가 온 시간은 ~" 문장으로 해석하세요.
4. `hour`의 계수는 양수입니다. `operating`에서 시간대별 평균 대여량을 구해 선그래프로 그리고, **시각을 숫자 하나로 넣는 직선 모델**이 시간대의 영향을 잘 설명하는지 쓰세요.

힌트: `dict(zip(X_train.columns, model.coef_))`, `groupby("hour")["rented_count"].mean()`, `plot(kind="line", marker="o")`.

**확인값** 평균 오차 326.2건, `is_rain` 계수 -536

In [ ]:
# 문제 4 · 코드 작성 1

In [ ]:
# 문제 4 · 코드 작성 2

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 5 · 의사결정나무로 예측하고 모델 비교하기

1. `DecisionTreeRegressor(max_depth=6, random_state=0)`를 네 입력으로 학습시키고 평가용 평균 오차를 구하세요.
2. `RandomForestRegressor(n_estimators=100, max_depth=6, random_state=0)`를 같은 네 입력으로 학습시키고 평가용 평균 오차를 구하세요.
3. 평균으로 예측, 기온 직선, 네 입력 선형 회귀, 의사결정나무, 랜덤 포레스트의 평균 오차를 한 표(`comparison`)로 정리하세요.
4. 평균으로 예측할 때에 비해 의사결정나무의 평균 오차가 몇 % 줄었는지 계산하세요.
5. 의사결정나무의 오차가 선형 회귀보다 작은 이유를 문제 4의 시간대별 그래프와 연결해 한 문장으로 쓰세요.
6. 랜덤 포레스트와 의사결정나무의 오차를 비교하고, 차이가 나는 이유를 "나무 여러 그루의 예측을 평균한다"는 점과 연결해 한 문장으로 쓰세요.

힌트: `DecisionTreeRegressor`, `RandomForestRegressor`, `pd.DataFrame({"모델": [...], "평균 오차(건)": [...]})`, 줄어든 비율 = (기준 오차 − 모델 오차) ÷ 기준 오차 × 100.
랜덤 포레스트도 `fit(입력, 정답)`으로 학습하고 `predict(입력)`으로 예측합니다.

**확인값** 의사결정나무 평균 오차 213.1건, 줄어든 비율 58.5%, 랜덤 포레스트 평균 오차 202.1건

In [ ]:
# 문제 5 · 코드 작성 1

In [ ]:
# 문제 5 · 코드 작성 2

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 6 · 나무의 깊이와 과적합

1. `max_depth`를 2, 4, 6, 8, 10, 제한 없음(`None`)으로 바꾸어 각각 학습시키고, 학습용 평균 오차와 평가용 평균 오차를 한 표(`depth_result`)로 정리하세요. 입력은 네 입력, `random_state=0`입니다.
2. 깊이가 깊어질수록 학습용 오차는 어떻게 변합니까? 깊이 제한이 없을 때 학습용과 평가용 오차를 비교해 무슨 일이 일어났는지 쓰세요.
3. 평가용 오차가 가장 작은 깊이를 표에서 찾으세요. 그 깊이를 그대로 선택해도 되는지, 학습용과 평가용 오차의 차이를 근거로 쓰세요.

힌트: 같은 계산을 깊이만 바꾸어 반복합니다. 괄호 부분을 채우세요.

```
rows = []
for depth in [2, 4, 6, 8, 10, None]:
    model = DecisionTreeRegressor(max_depth=depth, random_state=0)
    (학습시키기)
    rows.append({
        "깊이": depth,
        "학습용 평균 오차(건)": (학습용 평균 오차),
        "평가용 평균 오차(건)": (평가용 평균 오차),
    })
depth_result = pd.DataFrame(rows).round(1)
depth_result
```

**확인값** 깊이 제한 없음은 학습용 1.2건, 평가용 224.4건

In [ ]:
# 문제 6 · 코드 작성 1

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 7 · 많이 틀린 경우 찾아보기

문제 5의 나무(깊이 6)를 사용합니다.

1. 평가용 자료의 예측값을 구하고, 실제 대여량과 예측값의 차이(오차, 절댓값)가 큰 상위 5건을 입력 네 개, 실제 대여량, 예측값, 오차와 함께 표로 출력하세요.
2. 이 5건에 공통점이 있습니까?
3. 위 결과를 근거로 모델의 입력에 더하거나 바꾸면 좋을 것을 한 가지 쓰세요.

힌트:

```
predicted = tree.predict(X_test)
error = (y_test - predicted).abs()
table = X_test.assign(actual=y_test, predicted=predicted.round(0), error=error.round(0))
table.sort_values("error", ascending=False).head(5)
```

In [ ]:
# 문제 7 · 코드 작성 1

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 8 · 다른 기간에서 평가하기

지금까지는 날짜와 상관없이 무작위로 나누었습니다. 이번에는 **2017년 12월~2018년 8월 자료로 학습하고 2018년 9~11월 자료로 평가**합니다.

1. `operating`을 `date < "2018-09-01"`이면 학습 기간(`period_train`), 아니면 평가 기간(`period_test`)으로 나누고 행 수를 확인하세요.
2. 평균으로 예측, 선형 회귀(네 입력), 의사결정나무(깊이 6)를 학습 기간에서 학습시켜 평가 기간의 평균 오차를 구하고 한 표로 정리하세요.
3. 평가 기간은 어느 계절입니까? 무작위로 나누었던 문제 2·4·5의 결과와 비교해 오차가 어떻게 달라졌는지 한 문장으로 쓰세요.

힌트: 문제 2·4·5와 같은 방법을 학습 기간과 평가 기간 자료에 적용합니다. 날짜로 고르는 조건은 `operating["date"] < "2018-09-01"`, 입력은 `period_train[features]`, 계절은 `period_test["season"].unique()`로 확인합니다.

**확인값** 학습 기간 6,528행, 평가 기간 1,937행, 평균 오차는 평균 예측 513.9건, 선형 회귀 338.2건, 의사결정나무 291.1건

In [ ]:
# 문제 8 · 코드 작성 1

In [ ]:
# 문제 8 · 코드 작성 2

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용:

## 문제 9 · 내 모델 만들고 결과 보고하기

문제 5의 의사결정나무(깊이 6, 평가용 평균 오차 약 213건)보다 평가용 오차를 줄이는 입력 변수 조합을 직접 찾아봅니다. 입력은 `operating`의 **숫자 열**에서 고릅니다. 같은 분할(`random_state=0`)과 같은 모델(깊이 6 나무)을 사용합니다.

1. 입력 조합을 **두 가지 이상** 시도하고, 조합마다 평가용 평균 오차를 한 표로 정리하세요. 오차가 줄지 않은 조합도 표에 남기세요.
2. 가장 좋은 조합과 그 평균 오차를 쓰세요.
3. 운영 담당자에게 보고한다고 가정하고 **세 문장 이내**로 결론을 쓰세요. 이 모델로 예측할 수 있는 수준(오차 몇 건), 비교한 기준(평균으로 예측), 모델의 한계 한 가지를 넣으세요.

입력 후보는 `temperature`, `hour`, `humidity`, `wind_speed`, `solar_radiation`, `rainfall`, `snowfall`, `is_rain`, `weekday_number`입니다.

힌트: 조합을 딕셔너리로 모아 같은 계산을 반복합니다. 조합 이름과 열 이름 목록을 바꾸어 보세요.

```
combos = {
    "기본 네 입력": features,
    "내 조합 1": features + ["solar_radiation"],
}

rows = []
for name, cols in combos.items():
    Xa, Xb, ya, yb = train_test_split(operating[cols], operating["rented_count"], test_size=0.2, random_state=0)
    model = DecisionTreeRegressor(max_depth=6, random_state=0).fit(Xa, ya)
    rows.append({"조합": name, "평가용 평균 오차(건)": mean_absolute_error(yb, model.predict(Xb))})
pd.DataFrame(rows).round(1)
```

In [ ]:
# 문제 9 · 코드 작성 1

**분석 기록**

- 확인한 수치:
- 알게 된 점:
- 추가로 확인할 내용: